# Experiment 5: Deep Learning (CNN Baseline)

**Research Question:** How does learned representation (CNN) compare with handcrafted features (Classical ML)?

Instead of manually extracting shapes or textures, we use a Convolutional Neural Network (CNN) to automatically learn the most important hierarchical features from the raw pixels. We will fine-tune a pre-trained **ResNet18** model using PyTorch.

# Step 0: Google Colab Setup

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\nGoogle Drive Mounted successfully!")
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

# Step 1: Imports and Device Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using computational device: {device}")

# Step 2: Load Data via PyTorch DataLoaders
PyTorch's `ImageFolder` expects a specific folder structure. We will apply transformations to resize the images to 128x128, convert them to tensors, and normalize the pixel values.

In [ ]:
base_dir = '/content/drive/MyDrive/NeuroScan'

# --- Auto-detect for local execution ---
if 'base_dir' not in locals():
    current_dir = os.getcwd()
    base_dir = current_dir if os.path.exists(os.path.join(current_dir, 'data', 'Training')) else os.path.dirname(current_dir)

train_dir = os.path.join(base_dir, 'data', 'Training')
test_dir  = os.path.join(base_dir, 'data', 'Testing')

if not os.path.exists(train_dir):
    print(f"ERROR: The path {train_dir} does not exist! Please check your base_dir variable.")
else:
    # Data Transformations
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Datasets
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)

    # DataLoaders
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

    CLASS_NAMES = train_data.classes
    print(f"Classes mapped to indices: {train_data.class_to_idx}")
    print(f"Number of training images: {len(train_data)}")
    print(f"Number of testing images: {len(test_data)}")

# Step 3: Define the Model (ResNet18)
We load a pre-trained ResNet18 model and replace the final classification layer so it outputs exactly 4 classes instead of 1000.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze the early layers (optional, but speeds up training and prevents overfitting on small datasets)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer (this layer WILL be trained)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4) # 4 output classes for our tumors

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("Model architecture ready.")

# Step 4: Training Loop

In [ ]:
epochs = 5
print("Starting Training Loop...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
        # Calculate accuracy during training
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_data)
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}")

print("Training Complete!")

# Step 5: Evaluation on Test Set

In [ ]:
model.eval()
y_true = []
y_pred = []

print("Evaluating on testing data...")
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall    = recall_score(y_true, y_pred, average='weighted')
f1        = f1_score(y_true, y_pred, average='weighted')

print("\n================ EVALUATION METRICS ================")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("====================================================")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix: ResNet18 CNN')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()